# 13 — Henke Refraction vs Detector Parallax: Multi-Energy Disentangling

Two physical effects produce per-(hkl) DC shifts in the apparent
ring radius that look identical at single energy:

- **Detector parallax** — photons of different incidence angle
  penetrate to different depth in the sensor before being absorbed,
  shifting the apparent ring centroid by `d_eff · sin(2θ) / p_x`
  (energy-independent geometry).
- **Calibrant-grain refraction** — the X-ray index of refraction
  decrement `δ` causes a phase shift at each grain interface.  For
  CeO₂ at 63 keV via Henke tables, `δ ≈ 7 × 10⁻⁶`, scaling as
  `1/E²` (energy-dependent).

This notebook builds the analytical Cramér-Rao for the joint
identifiability of these two terms across single- and multi-energy
data.  The takeaway: at single energy they're correlated but
*not* fully degenerate (the per-2θ pattern of parallax differs
from the constant-per-ring pattern of refraction); at multi-energy
the `1/E²` Henke scaling breaks any residual ambiguity.


In [1]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import numpy as np

def two_theta_deg(a_A, hkl_norm, lam_A):
    return 2.0 * np.degrees(np.arcsin(lam_A * hkl_norm / (2.0 * a_A)))

def delta_henke(E_keV, E_ref_keV=63.0, delta_ref=7.0e-6):
    """CeO₂ Henke δ scales as 1/E² (Henke 1993 §3)."""
    return delta_ref * (E_ref_keV / E_keV) ** 2


def fisher_dimensionless(images, *, a_truth, p_x, hkl_norm,
                         d_eff_truth, delta_henke_ref,
                         sigma_R_px=0.05, n_per_ring=360):
    """Per-image Fisher in u-space for (L_sd_im, d_eff, δ_henke, a)."""
    N_im = len(images)
    n_p = N_im + 3
    n_rings = len(hkl_norm)
    rows_per_im = n_rings * n_per_ring
    n_total = N_im * rows_per_im
    J = np.zeros((n_total, n_p), dtype=np.float64)
    for im, spec in enumerate(images):
        L_sd = spec['L_sd_um']; lam = spec['lam_A']
        E_keV = 12.398 / lam
        delta_im = delta_henke(E_keV) / delta_henke_ref
        tt = two_theta_deg(a_truth, hkl_norm, lam)
        tt_rad = np.radians(tt); sec2 = 1.0 / np.cos(tt_rad) ** 2
        R_geom = L_sd * np.tan(tt_rad) / p_x
        col_L = np.repeat(R_geom, n_per_ring)
        col_d = np.repeat(np.sin(tt_rad) / p_x * d_eff_truth, n_per_ring)
        col_h = np.repeat(np.full_like(tt_rad, delta_im * L_sd / p_x * delta_henke_ref),
                            n_per_ring)
        dtt_da = -2.0 * np.tan(0.5 * tt_rad) / a_truth
        col_a = np.repeat(L_sd / p_x * sec2 * dtt_da, n_per_ring)
        rs = im * rows_per_im; re = rs + rows_per_im
        J[rs:re, im]      = col_L
        J[rs:re, N_im]    = col_d
        J[rs:re, N_im + 1] = col_h
        J[rs:re, N_im + 2] = col_a
    return (J.T @ J) / (sigma_R_px ** 2)


## Single-energy vs multi-energy


In [2]:
import math
a_truth = 5.4116; p_x = 150.0
d_eff_truth = -6_200.0; delta_henke_ref = 7.0e-6

hkl_sq = np.array([3,4,8,11,12,16,19,20,24,27,32,35,36,40,43,44,48,51,52,
                    56,59,64,67,68,72,75,76,80,83,84,88,91,96,99,100,104,107,108,
                    112,115,116,120,123,128,131,132,136,139,140], dtype=np.float64)
hkl_norm = np.sqrt(hkl_sq)
lam60 = 12.398 / 60.0; lam80 = 12.398 / 80.0; lam140 = 12.398 / 140.0
keep = (lam60 * hkl_norm / (2.0 * a_truth)) < 0.95
hkl_norm = hkl_norm[keep]

scenarios = [
    ('1 image @ 60 keV', [dict(L_sd_um=700_000.0, lam_A=lam60)]),
    ('2 energies (60 + 80 keV)', [
        dict(L_sd_um=700_000.0, lam_A=lam60),
        dict(L_sd_um=700_000.0, lam_A=lam80),
    ]),
    ('2 energies (60 + 140 keV; bigger lever)', [
        dict(L_sd_um=700_000.0, lam_A=lam60),
        dict(L_sd_um=700_000.0, lam_A=lam140),
    ]),
    ('4 energies (60, 80, 100, 140 keV)', [
        dict(L_sd_um=700_000.0, lam_A=lam60),
        dict(L_sd_um=700_000.0, lam_A=lam80),
        dict(L_sd_um=700_000.0, lam_A=12.398/100.0),
        dict(L_sd_um=700_000.0, lam_A=lam140),
    ]),
]

print(f'\n{"Scenario":<46s}  {"σ(d_eff) µm":>14s}  {"σ(δ_henke)":>16s}')
for label, ims in scenarios:
    F = fisher_dimensionless(ims, a_truth=a_truth, p_x=p_x,
                              hkl_norm=hkl_norm,
                              d_eff_truth=d_eff_truth,
                              delta_henke_ref=delta_henke_ref)
    cov = np.linalg.pinv(F)
    sigmas = np.sqrt(np.maximum(np.diag(cov), 0.0))
    sd_eff = sigmas[len(ims)] * d_eff_truth
    sd_h   = sigmas[len(ims) + 1] * delta_henke_ref
    print(f'  {label:<46s}  {sd_eff:>14.3e}  {sd_h:>16.3e}')



Scenario                                           σ(d_eff) µm        σ(δ_henke)
  1 image @ 60 keV                                    -3.343e+02         5.547e-07
  2 energies (60 + 80 keV)                            -2.242e+02         4.223e-07
  2 energies (60 + 140 keV; bigger lever)             -3.180e+02         5.350e-07
  4 energies (60, 80, 100, 140 keV)                   -1.986e+02         3.860e-07


## What this means

The data already supports separate identification of d_eff and δ_henke
even at **single energy** (the per-(hkl) DC patterns of the two
contributions differ).  Multi-energy data with sufficient lever-arm
modestly tightens both σ values (~1.5× at 2 energies).

The current v2 spec exposes only the single `Parallax` parameter
(an effective scalar that absorbs both physical contributions).
**Adding a separate `δ_henke` parameter to the spec** would let the
LM disentangle them — the data already supports it.  This is
identified as straightforward future work.

## See also

- [`dev/paper/runners/run_henke_disentangle.py`](../dev/paper/runners/run_henke_disentangle.py)
- §"three fixes" prose in the paper (the F1/F2/F1+F2 table)
- Henke, Gullikson & Davis 1993 — the canonical X-ray refraction tables
